In [1]:
!pip install requests

In [2]:
UTSA_API_KEY = "utsa-jABQlGLaTrae2bqMHyAvPxTvE9KTP0DEWYIXhvtgkDkVcGjp44rN6G56x1aGiyem"
UTSA_BASE_URL = "http://149.165.173.247:8888/v1"
UTSA_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

In [3]:
import requests
import re
import time

def call_llm(prompt):
    url = f"{UTSA_BASE_URL}/chat/completions"

    headers = {
        "Authorization": f"Bearer {UTSA_API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": UTSA_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }

    try:
        res = requests.post(url, headers=headers, json=data, timeout=60)
        response = res.json()

        if "choices" not in response:
            return f"API ERROR: {response}"

        output = response["choices"][0]["message"]["content"]
        output = re.sub(r"<think>.*?</think>", "", output, flags=re.DOTALL)

        return output.strip()

    except Exception as e:
        print("Retrying due to error:", e)
        time.sleep(5)
        return "ERROR"

In [4]:
def debater_a(question, context=""):
    prompt = f"""
You are Debater A (Proponent).
Argue YES with reasoning.

Question: {question}
Context: {context}
"""
    return call_llm(prompt)


def debater_b(question, context=""):
    prompt = f"""
You are Debater B (Opponent).
Argue NO and counter Debater A.

Question: {question}
Context: {context}
"""
    return call_llm(prompt)

In [5]:
def judge(question, debate):
    prompt = f"""
You are a judge.

Question: {question}

Debate:
{debate}

Give:
1. Analysis
2. Winner
3. Final Answer (YES or NO)
4. Confidence (1-5)
"""
    return call_llm(prompt)

In [6]:
def run_debate(question, rounds=2):
    log = ""

    a = debater_a(question)
    b = debater_b(question)

    log += f"A: {a}\nB: {b}\n"

    for i in range(rounds):
        a = debater_a(question, log)
        b = debater_b(question, log)

        log += f"\nRound {i+1} A: {a}\n"
        log += f"Round {i+1} B: {b}\n"

    result = judge(question, log)

    return log, result

In [7]:
question = "Did the Roman Empire exist at the same time as the Mayan civilization?"

debate, result = run_debate(question)

print("DEBATE:\n", debate)
print("\nRESULT:\n", result)

DEBATE:
 A: I'm happy to present my argument in favor of the Roman Empire and Mayan civilization coexisting.

Yes, the Roman Empire and the Mayan civilization did exist at the same time.

The Roman Empire, which was a vast and powerful state, reached its peak in the 1st and 2nd centuries AD. This was during the Pax Romana, a period of relative peace and stability under the rule of emperors such as Augustus, Trajan, and Hadrian.

On the other hand, the Mayan civilization flourished in Central America, primarily in present-day Mexico, Guatemala, Belize, and Honduras. The Classic Period of the Mayan civilization, which is often considered the peak of their power and culture, spanned from approximately 200 to 900 AD.

Given this timeline, it's clear that the Roman Empire and the Mayan civilization did overlap. In fact, the Roman Empire was already in decline when the Mayan civilization reached its peak. However, the two civilizations did coexist for a significant period.

The Roman Empire'

In [8]:
def multi_judge(question, debate, num_judges=3):
    results = []

    for _ in range(num_judges):
        results.append(judge(question, debate))

    return results

In [9]:
def final_decision(judgements):
    yes_count = sum("YES" in j.upper() for j in judgements)
    no_count = sum("NO" in j.upper() for j in judgements)

    if yes_count > no_count:
        return "FINAL DECISION: YES"
    elif no_count > yes_count:
        return "FINAL DECISION: NO"
    return "FINAL DECISION: TIE"

In [10]:
debate, _ = run_debate(question)

judgements = multi_judge(question, debate)

for i, j in enumerate(judgements):
    print(f"\nJudge {i+1}:\n", j)

print("\n", final_decision(judgements))


Judge 1:
 **Analysis**

The debate between Debater A and Debater B revolves around the question of whether the Roman Empire and the Mayan civilization coexisted in time. Both debaters present strong arguments, but ultimately, Debater B's argument is more convincing.

Debater A's argument relies heavily on a simplistic timeline that fails to account for the complexities of historical events. While it is true that the Roman Empire and the Mayan civilization had some overlap in their periods of significant growth and influence, Debater A's argument ignores the fact that the Mayan civilization had already established itself as a well-established civilization before the Roman Empire rose to power.

Debater B, on the other hand, presents a more nuanced understanding of the historical events. They argue that the Mayan civilization had already established its cities and kingdoms in the Preclassic Period (2000 BC-200 AD), which predates the Roman Empire's rise to power. They also point out tha

In [13]:
questions = [
"Is the Earth flat?","Do vaccines prevent diseases?","Is the sun a star?","Is water H2O?",
"Can humans survive without oxygen?","Is fire hot?","Do birds fly?","Is ice cold?",
"Is sugar sweet?","Can fish swim?","Is gravity real?","Is Python a programming language?",
"Is AI useful?","Can humans live on Mars?","Is electricity dangerous?","Is water wet?",
"Do plants need sunlight?","Is time travel possible?","Is the ocean salty?",
"Do humans need sleep?","Is the sky blue?","Is math important?","Is oxygen necessary?",
"Do dogs bark?","Can humans fly naturally?","Is the internet useful?",
"Is the brain important?","Is the heart an organ?","Can robots replace humans?",
"Is learning important?","Is data valuable?","Is coding useful?","Is AI dangerous?",
"Is the Earth round?","Is science important?","Do humans age?","Is the sun hot?",
"Is water essential?","Do computers process data?","Is space vast?","Is logic important?",
"Is reasoning useful?","Is knowledge power?","Is innovation important?",
"Is health important?","Is education important?","Is technology advancing?",
"Is communication important?","Is teamwork valuable?","Is creativity important?"
]

In [14]:
ground_truth = ["YES"] * len(questions)

In [15]:
def direct_qa(question):
    prompt = f"""
Answer YES or NO with reasoning.

Question: {question}
"""
    return call_llm(prompt)

In [16]:
results = []

for i, q in enumerate(questions):
    print(f"\nQ{i+1}: {q}")

    try:
        ans = direct_qa(q)

        results.append({
            "question": q,
            "answer": ans,
            "truth": ground_truth[i]
        })

        time.sleep(2)

    except Exception as e:
        print("Error:", e)
        time.sleep(5)


Q1: Is the Earth flat?

Q2: Do vaccines prevent diseases?

Q3: Is the sun a star?

Q4: Is water H2O?

Q5: Can humans survive without oxygen?

Q6: Is fire hot?

Q7: Do birds fly?

Q8: Is ice cold?

Q9: Is sugar sweet?

Q10: Can fish swim?

Q11: Is gravity real?

Q12: Is Python a programming language?

Q13: Is AI useful?

Q14: Can humans live on Mars?

Q15: Is electricity dangerous?

Q16: Is water wet?

Q17: Do plants need sunlight?

Q18: Is time travel possible?

Q19: Is the ocean salty?

Q20: Do humans need sleep?

Q21: Is the sky blue?

Q22: Is math important?

Q23: Is oxygen necessary?

Q24: Do dogs bark?

Q25: Can humans fly naturally?

Q26: Is the internet useful?

Q27: Is the brain important?

Q28: Is the heart an organ?

Q29: Can robots replace humans?

Q30: Is learning important?

Q31: Is data valuable?

Q32: Is coding useful?

Q33: Is AI dangerous?

Q34: Is the Earth round?

Q35: Is science important?

Q36: Do humans age?

Q37: Is the sun hot?

Q38: Is water essential?

Q39: D

In [17]:
def extract(text):
    t = text.upper()
    if "YES" in t: return "YES"
    if "NO" in t: return "NO"
    return "UNKNOWN"

correct = 0

for r in results:
    if extract(r["answer"]) == r["truth"]:
        correct += 1

print("Accuracy:", correct/len(results))

Accuracy: 0.84


In [18]:
for q in questions[:3]:
    print("\nQUESTION:", q)
    _, res = run_debate(q)
    print(res)
    time.sleep(6)


QUESTION: Is the Earth flat?
**Analysis**

The debate on whether the Earth is flat or not has been ongoing for centuries, with proponents of both sides presenting their arguments. In this debate, Debater A presents several lines of evidence that confirm the Earth is a sphere, including the curvature of the Earth, satellite imagery, the existence of time zones, and the phenomenon of ships' shadows on the ocean's surface. Debater B, on the other hand, presents several counterarguments that challenge Debater A's claims, including the idea that the Earth's curvature is not uniform, that satellite imagery and spacecraft observations are distorted or manipulated, and that the existence of time zones can be explained by a rotating flat Earth.

Throughout the debate, both debaters present valid points, but Debater A's arguments are more convincing and well-supported by scientific evidence. Debater A presents a wide range of evidence from multiple fields of science, including astronomy, physic